# T09 — Query Techniques

The baseline embeds the user query as is. If its wording differs from the documents, relevant chunks can rank too low. Two techniques change the query side only, without touching the vector store or the LLM:

| Technique | Idea |
|---|---|
| **Query expansion** | Generate N rephrasings, retrieve for each, merge the rankings with RRF |
| **HyDE** | Let the LLM draft a hypothetical answer and retrieve with it as an extra query |

All three agents below answer the same question; the last cell compares the source files they retrieved.

## Setup

Run T01 first: it builds the vector store at `backend/db/data_vs.db`. If the store is empty, `load_or_build_vector_store()` builds it from all documents, which takes several minutes.

In [ ]:
import sys

from loguru import logger

from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    ask,
    build_agent,
    build_llm,
    load_or_build_vector_store,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
llm = build_llm("openai")

QUERY = "Which pallets in our portfolio have a third-party verified EPD?"

## Baseline

In [ ]:
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)
baseline = await ask(agent, QUERY)

## Query expansion

The LLM generates `N` rephrasings of the query. Each one is embedded and retrieved separately, and the rankings are merged with Reciprocal Rank Fusion (RRF), the same merge used for hybrid retrieval in T07.

```
Original query ──► embed ──► top-k list 0 ─┐
Rephrasing 1   ──► embed ──► top-k list 1 ─┼─► RRF ──► final top-k
...                                        │
Rephrasing N   ──► embed ──► top-k list N ─┘
```

Cost: one extra LLM call and N extra embedding calls per query. Rephrasings can drift from the original intent; the DEBUG log below shows them.

In [ ]:
expansion_agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=5,
)

sink = logger.add(
    sys.stderr,
    level="DEBUG",
    filter="conversational_toolkit.utils.retriever",
    format="{message}",
)
expansion = await ask(expansion_agent, QUERY)
logger.remove(sink)

## HyDE: Hypothetical Document Embeddings

A question and the passage that answers it are worded differently, so their embeddings are not as close as they could be. HyDE lets the LLM draft a hypothetical answer without any context. A wrong answer is fine, as long as it reads like the documents. The draft is embedded **in addition to** the query, and both rankings are merged with RRF.

```
Query ─────────────────────────────────────────► embed ──► top-k ─┐
Query ──► LLM (no context) ──► hypothetical answer ──► embed ──► top-k ─┴─► RRF
```

Cost: one extra LLM call and one extra embedding call per query. If the draft goes in the wrong direction, it pulls in unrelated chunks.

In [ ]:
hyde_agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
    enable_hyde=True,
)

sink = logger.add(
    sys.stderr,
    level="DEBUG",
    filter="conversational_toolkit.utils.retriever",
    format="{message}",
)
hyde = await ask(hyde_agent, QUERY)
logger.remove(sink)

## Comparison

In [ ]:
for name, answer in [("Baseline", baseline), ("Expansion", expansion), ("HyDE", hyde)]:
    print(f"{name}: {sorted({s.metadata['source_file'] for s in answer.sources})}")